# EXAMEN PRÁCTICO DE LA ASIGNATURA: OPTIMIZACIÓN

**Alumno:** Juan Elías Lara Soto  
**Titulación:** Doble Grado en Matemáticas e Ingeniería Informática  
**Universidad:** Universidad de Málaga  
**Fecha:** 15/12/2025


Considera un mercado eléctrico compuesto por $N_{G}$ productores de electricidad que han de satisfacer una demanda $D$. El coste de producción de cada productor viene dada por una *función lineal por tramos o a trozos*, caracterizada por un número de tramos $N_{p_{i}}$,  $\forall i = 1, \ldots, N_{G}$) y sus correspondientes longitudes y pendientes $(l_{i,j}, m_{i,j}), \forall i = 1, \ldots, N_{G}; \forall j = 1, \ldots, N_{p_{i}}$.

No obstante, estas funciones presentan una "zona muerta o prohibida", exisitiendo una producción mínima $\underline{P}_i$ tal que el productor $i$ no puede producir en el intervalo abierto $(0, \underline{P}_i)$. Asimismo, producir a producción mínima acarrea un coste fijo $c_i$, de tal forma que el primer tramo variable de la función de costes parte del punto $(\underline{P}_i, c_i)$, $\forall i = 1, \ldots, N_{G}$.

Se pide:
1.   Formular un modelo de Programación Matemática para minimizar el coste total en el que incurre el mercado para satisfacer la demanda de electricidad $D$. Para ello, asume que la parte de la función de coste de los productores formada por los tramos lineales de coste variable es convexa (2.5 puntos).
2.   Implementar el modelo anterior en AMPL (1.5 puntos).
3.  Considera el caso particular en que el mercado eléctrico está formado por cinco productores ($N_{G} = 5$) cuyas funciones de coste vienen dadas en la siguiente tabla:  

     <center>

     |  Productor    | Tramo 1   | Tramo 2  | Tramo 3 |    $\underline{P}$ | c |
     | ------------  | --------     | ---      | ---    |  --- |---|
     | 1            | (20, 10)      | (20, 15) | (30, 18)| 10   |100 |
     | 2            | (10, 5)       | (50, 13) | (60, 17)|  5  | 400 |
     | 3            | (100, 16)     | (50, 16) | (50, 31)|   10  |320 |
     | 4            | (80, 15)      | (55, 17) | (45, 31)|   5  |300 |
     | 5            | (90, 14)      | (40, 18) | (50, 33)|   7  |350 |

     </center>

     donde cada par $(l_{i,j}, m_{i,j})$ con $i= 1, \ldots, 5$, y $j = 1, 2, 3$ en la tabla especifica la longitud o anchura del trozo lineal $l_{i,j}$ junto con su pendiente $m_{i,j}$. El mercado debe satisfacer una demanda $D = 201$.

     *   Resuelve este caso particular con un solver para programas lineales enteros (1 punto).
     *   Resuelve este caso particular mediante el *algoritmo de ramificación y acotación*, utilizando un solver adecuado para resolver los problemas lineales relajados que sean necesarios. Representa el árbol al que la aplicación del algoritmo ha dado lugar, especificando los tipos de podas que has realizado en cada caso hasta certificar la solución óptima del problema entero (5 puntos).

In [1]:
%pip install -q amplpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 25.4 MB/s eta 0:00:00


In [76]:
# Integración en Google Colab
from amplpy import AMPL, ampl_notebook

Primal = ampl_notebook(
    modules=["highs", "cplex"],  # Solvers que queremos instalar
    license_uuid="062f1a26-719d-4062-9a14-f91b8a2a0c4c",
)

Licensed to Bundle #7272.7817 expiring 20260228: 5003302 - Optimization; 5003408 - Operations Research, Prof. Juan Miguel Morales Gonz?lez, University of Malaga.


2.Implementar el modelo anterior en AMPL (1.5 puntos).

In [77]:
Primal.eval(r"""
reset;
set Prod;
set Tram;

param D >= 0;

param Pmin {Prod} >= 0;         # Producción mínima (zona muerta/prohibida)
param c    {Prod} >= 0;         # Coste fijo al producir

param lon   {Prod, Tram} >= 0;  # Longitud de cada tramo (variable)
param slope {Prod, Tram} >= 0;  # Pendiente de cada tramo (coste marginal)

var y {i in Prod} binary;       # 1 si el productor i produce
var p {i in Prod, j in Tram} >= 0;  # Producción variable por tramo

minimize coste_total:
    sum{i in Prod} c[i]*y[i] + sum{i in Prod, j in Tram} slope[i,j]*p[i,j];

s.t. balance:
    sum{i in Prod} Pmin[i]*y[i] + sum{i in Prod, j in Tram} p[i,j] = D;
s.t. limite_tramo {i in Prod, j in Tram}:
    p[i,j] <= lon[i,j] * y[i];
""")

3.Resuelve este caso particular con un solver para programas lineales enteros (1 punto).

In [78]:
def preparar_datos_mercado_ext():
    import pandas as pd
    import numpy as np


    tramos = ["T1", "T2", "T3"]
    productores = ["G1", "G2", "G3", "G4", "G5"]


    demanda = 201


    Pmin = {
        "G1": 10,
        "G2": 5,
        "G3": 10,
        "G4": 5,
        "G5": 7,
    }

    c = {
        "G1": 100,
        "G2": 400,
        "G3": 320,
        "G4": 300,
        "G5": 350,
    }


    lon_df = pd.DataFrame(
        np.array(
            [
                [20, 20, 30],
                [10, 50, 60],
                [100, 50, 50],
                [80, 55, 45],
                [90, 40, 50],
            ]
        ),
        columns=tramos,
        index=productores,
    )

    slope_df = pd.DataFrame(
        np.array(
            [
                [10, 15, 18],
                [5, 13, 17],
                [16, 16, 31],
                [15, 17, 31],
                [14, 18, 33],
            ]
        ),
        columns=tramos,
        index=productores,
    )

    return demanda, tramos, productores, Pmin, c, lon_df, slope_df


In [79]:
# Preparamos los datos
demanda, tramos, productores, Pmin, c, lon_df, slope_df = preparar_datos_mercado_ext()

# Conjuntos
Primal.set["Prod"] = productores
Primal.set["Tram"] = tramos

# Parámetros
Primal.get_parameter("D").set(demanda)
Primal.get_parameter("Pmin").set_values(Pmin)
Primal.get_parameter("c").set_values(c)

Primal.get_parameter("lon").set_values(lon_df)
Primal.get_parameter("slope").set_values(slope_df)



In [80]:
Primal.solve(solver="cplex") # Resolvemos con el solver "cplex"
assert Primal.solve_result == "solved"  # Comprobamos que el problema se ha resuelto correctamente

CPLEX 22.1.2: CPLEX 22.1.2: optimal solution; objective 3145
11 simplex iterations


In [96]:
import pandas as pd

y = Primal.get_variable("y")
df_y = y.get_values().to_pandas()
print(df_y)

p = Primal.get_variable("p")
df_p = p.get_values().to_pandas()



# Producción por tramos (tabla tal cual)
df_tramos = df_p.copy()
df_tramos.rename(columns={"p.val": "p"}, inplace=True)

# Pasamos a formato ancho: un productor por fila, un tramo por columna
df_tramos_wide = df_tramos.reset_index().pivot(
    index="index0", columns="index1", values="p"
)
df_tramos_wide.columns.name = None

# Producción mínima efectiva
Pmin_df = Primal.get_parameter("Pmin").get_values().to_pandas()
Pmin_series = Pmin_df.iloc[:, 0]
y_series = df_y["y.val"]
df_tramos_wide["Pmin"] = Pmin_series * y_series

# Producción total
df_tramos_wide["Total"] = (
    df_tramos_wide[["T1", "T2", "T3"]].sum(axis=1)
    + df_tramos_wide["Pmin"]
)

print(df_tramos_wide)
print(df_tramos_wide["Total"].sum())

valor_objetivo = Primal.get_objective("coste_total").value()
print(f"\nValor objetivo (coste total): {valor_objetivo:.3f}")

    y.val
G1      1
G2      1
G3      0
G4      0
G5      1
        T1  T2  T3  Pmin  Total
index0                         
G1      20   9   0    10     39
G2      10  50   0     5     65
G3       0   0   0     0      0
G4       0   0   0     0      0
G5      90   0   0     7     97
201

Valor objetivo (coste total): 3145.000


3. Resuelve este caso particular mediante el algoritmo de ramificación y acotación, utilizando un solver adecuado para resolver los problemas lineales relajados que sean necesarios. Representa el árbol al que la aplicación del algoritmo ha dado lugar, especificando los tipos de podas que has realizado en cada caso hasta certificar la solución óptima del problema entero (5 puntos).

In [82]:
# Integración en Google Colab
from amplpy import AMPL, ampl_notebook

Pr = ampl_notebook(
    modules=["highs", "cplex"],  # Solvers que queremos instalar
    license_uuid="062f1a26-719d-4062-9a14-f91b8a2a0c4c",
)

Licensed to Bundle #7272.7817 expiring 20260228: 5003302 - Optimization; 5003408 - Operations Research, Prof. Juan Miguel Morales Gonz?lez, University of Malaga.


In [83]:
Pr.eval(r"""
reset;
set Prod;
set Tram;

param D >= 0;

param Pmin {Prod} >= 0;         # Producción mínima (zona muerta/prohibida)
param c    {Prod} >= 0;         # Coste fijo al producir

param lon   {Prod, Tram} >= 0;  # Longitud de cada tramo (variable)
param slope {Prod, Tram} >= 0;  # Pendiente de cada tramo (coste marginal)

var y {i in Prod} >=0,<=1;       # 1 si el productor i produce
var p {i in Prod, j in Tram} >= 0;  # Producción variable por tramo

minimize coste_total:
    sum{i in Prod} c[i]*y[i] + sum{i in Prod, j in Tram} slope[i,j]*p[i,j];

s.t. balance:
    sum{i in Prod} Pmin[i]*y[i] + sum{i in Prod, j in Tram} p[i,j] = D;
s.t. limite_tramo {i in Prod, j in Tram}:
    p[i,j] <= lon[i,j] * y[i];
""")

In [84]:
# Cargar datos desde la función
demanda, tramos, productores, Pmin, c, lon_df, slope_df = preparar_datos_mercado_ext()

# Conjuntos
Pr.set["Prod"] = productores
Pr.set["Tram"] = tramos

# Parámetros
Pr.get_parameter("D").set(demanda)
Pr.get_parameter("Pmin").set_values(Pmin)
Pr.get_parameter("c").set_values(c)

Pr.get_parameter("lon").set_values(lon_df)
Pr.get_parameter("slope").set_values(slope_df)



In [85]:
Pr.solve(solver="cplex") # Resolvemos con el solver "cplex"
assert Pr.solve_result == "solved"  # Comprobamos que el problema se ha resuelto correctamente

CPLEX 22.1.2: CPLEX 22.1.2: optimal solution; objective 3123.846154
9 simplex iterations


In [97]:
import pandas as pd

y = Pr.get_variable("y")
df_y = y.get_values().to_pandas()
print(df_y)

p = Pr.get_variable("p")
df_p = p.get_values().to_pandas()



# Producción por tramos (tabla tal cual)
df_tramos = df_p.copy()
df_tramos.rename(columns={"p.val": "p"}, inplace=True)

# Pasamos a formato ancho: un productor por fila, un tramo por columna
df_tramos_wide = df_tramos.reset_index().pivot(
    index="index0", columns="index1", values="p"
)
df_tramos_wide.columns.name = None

# Producción mínima efectiva
Pmin_df = Pr.get_parameter("Pmin").get_values().to_pandas()
Pmin_series = Pmin_df.iloc[:, 0]
y_series = df_y["y.val"]
df_tramos_wide["Pmin"] = Pmin_series * y_series

# Producción total
df_tramos_wide["Total"] = (
    df_tramos_wide[["T1", "T2", "T3"]].sum(axis=1)
    + df_tramos_wide["Pmin"]
)

print(df_tramos_wide)
print(df_tramos_wide["Total"].sum())
valor_objetivo = Pr.get_objective("coste_total").value()
print(f"\nValor objetivo (coste total): {valor_objetivo:.3f}")




       y.val
G1  1.000000
G2  0.830769
G3  0.000000
G4  0.000000
G5  1.000000
               T1         T2   T3       Pmin  Total
index0                                             
G1      20.000000  20.000000  0.0  10.000000   50.0
G2       8.307692  41.538462  0.0   4.153846   54.0
G3       0.000000   0.000000  0.0   0.000000    0.0
G4       0.000000   0.000000  0.0   0.000000    0.0
G5      90.000000   0.000000  0.0   7.000000   97.0
201.0

Valor objetivo (coste total): 3123.846


ramifico por y2

In [100]:
# Integración en Google Colab
from amplpy import AMPL, ampl_notebook

P1 = ampl_notebook(
    modules=["highs", "cplex"],  # Solvers que queremos instalar
    license_uuid="062f1a26-719d-4062-9a14-f91b8a2a0c4c",
)

Licensed to Bundle #7272.7817 expiring 20260228: 5003302 - Optimization; 5003408 - Operations Research, Prof. Juan Miguel Morales Gonz?lez, University of Malaga.


In [101]:
P1.eval(r"""
reset;
set Prod;
set Tram;

param D >= 0;

param Pmin {Prod} >= 0;         # Producción mínima (zona muerta/prohibida)
param c    {Prod} >= 0;         # Coste fijo al producir

param lon   {Prod, Tram} >= 0;  # Longitud de cada tramo (variable)
param slope {Prod, Tram} >= 0;  # Pendiente de cada tramo (coste marginal)

var y {i in Prod} >=0,<=1;       # 1 si el productor i produce
var p {i in Prod, j in Tram} >= 0;  # Producción variable por tramo

minimize coste_total:
    sum{i in Prod} c[i]*y[i] + sum{i in Prod, j in Tram} slope[i,j]*p[i,j];

s.t. balance:
    sum{i in Prod} Pmin[i]*y[i] + sum{i in Prod, j in Tram} p[i,j] = D;
s.t. limite_tramo {i in Prod, j in Tram}:
    p[i,j] <= lon[i,j] * y[i];
bounding1: y["G2"]<=0;
""")

In [102]:
# Cargar datos desde la función
demanda, tramos, productores, Pmin, c, lon_df, slope_df = preparar_datos_mercado_ext()

# Conjuntos
P1.set["Prod"] = productores
P1.set["Tram"] = tramos

# Parámetros
P1.get_parameter("D").set(demanda)
P1.get_parameter("Pmin").set_values(Pmin)
P1.get_parameter("c").set_values(c)

P1.get_parameter("lon").set_values(lon_df)
P1.get_parameter("slope").set_values(slope_df)



In [103]:
P1.solve(solver="cplex") # Resolvemos con el solver "cplex"
assert P1.solve_result == "solved"  # Comprobamos que el problema se ha resuelto correctamente

CPLEX 22.1.2: CPLEX 22.1.2: optimal solution; objective 3128
7 simplex iterations


In [99]:
import pandas as pd

y = P1.get_variable("y")
df_y = y.get_values().to_pandas()
print(df_y)

p = P1.get_variable("p")
df_p = p.get_values().to_pandas()



# Producción por tramos (tabla tal cual)
df_tramos = df_p.copy()
df_tramos.rename(columns={"p.val": "p"}, inplace=True)

# Pasamos a formato ancho: un productor por fila, un tramo por columna
df_tramos_wide = df_tramos.reset_index().pivot(
    index="index0", columns="index1", values="p"
)
df_tramos_wide.columns.name = None

# Producción mínima efectiva
Pmin_df = P1.get_parameter("Pmin").get_values().to_pandas()
Pmin_series = Pmin_df.iloc[:, 0]
y_series = df_y["y.val"]
df_tramos_wide["Pmin"] = Pmin_series * y_series

# Producción total
df_tramos_wide["Total"] = (
    df_tramos_wide[["T1", "T2", "T3"]].sum(axis=1)
    + df_tramos_wide["Pmin"]
)

print(df_tramos_wide)
print(df_tramos_wide["Total"].sum())
valor_objetivo = P1.get_objective("coste_total").value()
print(f"\nValor objetivo (coste total): {valor_objetivo:.3f}")


     y.val
G1  1.0000
G2  0.0000
G3  0.3375
G4  0.0000
G5  1.0000
           T1      T2   T3    Pmin  Total
index0                                   
G1      20.00  20.000  0.0  10.000   50.0
G2       0.00   0.000  0.0   0.000    0.0
G3      33.75  16.875  0.0   3.375   54.0
G4       0.00   0.000  0.0   0.000    0.0
G5      90.00   0.000  0.0   7.000   97.0
201.0

Valor objetivo (coste total): 3128.000


In [110]:
# Integración en Google Colab
from amplpy import AMPL, ampl_notebook

P2 = ampl_notebook(
    modules=["highs", "cplex"],  # Solvers que queremos instalar
    license_uuid="062f1a26-719d-4062-9a14-f91b8a2a0c4c",
)

Licensed to Bundle #7272.7817 expiring 20260228: 5003302 - Optimization; 5003408 - Operations Research, Prof. Juan Miguel Morales Gonz?lez, University of Malaga.


In [115]:
P2.eval(r"""
reset;
set Prod;
set Tram;

param D >= 0;

param Pmin {Prod} >= 0;         # Producción mínima (zona muerta/prohibida)
param c    {Prod} >= 0;         # Coste fijo al producir

param lon   {Prod, Tram} >= 0;  # Longitud de cada tramo (variable)
param slope {Prod, Tram} >= 0;  # Pendiente de cada tramo (coste marginal)

var y {i in Prod} >=0,<=1;       # 1 si el productor i produce
var p {i in Prod, j in Tram} >= 0;  # Producción variable por tramo

minimize coste_total:
    sum{i in Prod} c[i]*y[i] + sum{i in Prod, j in Tram} slope[i,j]*p[i,j];

s.t. balance:
    sum{i in Prod} Pmin[i]*y[i] + sum{i in Prod, j in Tram} p[i,j] = D;
s.t. limite_tramo {i in Prod, j in Tram}:
    p[i,j] <= lon[i,j] * y[i];
bounding1: y["G2"]>=1;
""")

In [116]:
# Cargar datos desde la función
demanda, tramos, productores, Pmin, c, lon_df, slope_df = preparar_datos_mercado_ext()

# Conjuntos
P2.set["Prod"] = productores
P2.set["Tram"] = tramos

# Parámetros
P2.get_parameter("D").set(demanda)
P2.get_parameter("Pmin").set_values(Pmin)
P2.get_parameter("c").set_values(c)

P2.get_parameter("lon").set_values(lon_df)
P2.get_parameter("slope").set_values(slope_df)



In [117]:
P2.solve(solver="cplex") # Resolvemos con el solver "cplex"
assert P2.solve_result == "solved"  # Comprobamos que el problema se ha resuelto correctamente

CPLEX 22.1.2: CPLEX 22.1.2: optimal solution; objective 3127.42268
7 simplex iterations


In [118]:
import pandas as pd

y = P2.get_variable("y")
df_y = y.get_values().to_pandas()
print(df_y)

p = P2.get_variable("p")
df_p = p.get_values().to_pandas()



# Producción por tramos (tabla tal cual)
df_tramos = df_p.copy()
df_tramos.rename(columns={"p.val": "p"}, inplace=True)

# Pasamos a formato ancho: un productor por fila, un tramo por columna
df_tramos_wide = df_tramos.reset_index().pivot(
    index="index0", columns="index1", values="p"
)
df_tramos_wide.columns.name = None

# Producción mínima efectiva
Pmin_df = P2.get_parameter("Pmin").get_values().to_pandas()
Pmin_series = Pmin_df.iloc[:, 0]
y_series = df_y["y.val"]
df_tramos_wide["Pmin"] = Pmin_series * y_series

# Producción total
df_tramos_wide["Total"] = (
    df_tramos_wide[["T1", "T2", "T3"]].sum(axis=1)
    + df_tramos_wide["Pmin"]
)

print(df_tramos_wide)
print(df_tramos_wide["Total"].sum())
valor_objetivo = P2.get_objective("coste_total").value()
print(f"\nValor objetivo (coste total): {valor_objetivo:.3f}")


       y.val
G1  1.000000
G2  1.000000
G3  0.000000
G4  0.000000
G5  0.886598
               T1    T2   T3       Pmin  Total
index0                                        
G1      20.000000  20.0  0.0  10.000000   50.0
G2      10.000000  50.0  0.0   5.000000   65.0
G3       0.000000   0.0  0.0   0.000000    0.0
G4       0.000000   0.0  0.0   0.000000    0.0
G5      79.793814   0.0  0.0   6.206186   86.0
201.0

Valor objetivo (coste total): 3127.423


In [119]:
# Integración en Google Colab
from amplpy import AMPL, ampl_notebook

P3 = ampl_notebook(
    modules=["highs", "cplex"],  # Solvers que queremos instalar
    license_uuid="062f1a26-719d-4062-9a14-f91b8a2a0c4c",
)

Licensed to Bundle #7272.7817 expiring 20260228: 5003302 - Optimization; 5003408 - Operations Research, Prof. Juan Miguel Morales Gonz?lez, University of Malaga.


In [144]:
P3.eval(r"""
reset;
set Prod;
set Tram;

param D >= 0;

param Pmin {Prod} >= 0;         # Producción mínima (zona muerta/prohibida)
param c    {Prod} >= 0;         # Coste fijo al producir

param lon   {Prod, Tram} >= 0;  # Longitud de cada tramo (variable)
param slope {Prod, Tram} >= 0;  # Pendiente de cada tramo (coste marginal)

var y {i in Prod} >=0,<=1;       # 1 si el productor i produce
var p {i in Prod, j in Tram} >= 0;  # Producción variable por tramo

minimize coste_total:
    sum{i in Prod} c[i]*y[i] + sum{i in Prod, j in Tram} slope[i,j]*p[i,j];

s.t. balance:
    sum{i in Prod} Pmin[i]*y[i] + sum{i in Prod, j in Tram} p[i,j] = D;
s.t. limite_tramo {i in Prod, j in Tram}:
    p[i,j] <= lon[i,j] * y[i];
bounding1: y["G2"]>=1;
bounding2: y["G5"]=>1;

""")

In [145]:
# Cargar datos desde la función
demanda, tramos, productores, Pmin, c, lon_df, slope_df = preparar_datos_mercado_ext()

# Conjuntos
P3.set["Prod"] = productores
P3.set["Tram"] = tramos

# Parámetros
P3.get_parameter("D").set(demanda)
P3.get_parameter("Pmin").set_values(Pmin)
P3.get_parameter("c").set_values(c)

P3.get_parameter("lon").set_values(lon_df)
P3.get_parameter("slope").set_values(slope_df)



In [146]:
P3.solve(solver="cplex") # Resolvemos con el solver "cplex"
assert P3.solve_result == "solved"  # Comprobamos que el problema se ha resuelto correctamente

CPLEX 22.1.2: CPLEX 22.1.2: optimal solution; objective 3162
6 simplex iterations


In [147]:
import pandas as pd

y = P3.get_variable("y")
df_y = y.get_values().to_pandas()
print(df_y)

p = P1.get_variable("p")
df_p = p.get_values().to_pandas()



# Producción por tramos (tabla tal cual)
df_tramos = df_p.copy()
df_tramos.rename(columns={"p.val": "p"}, inplace=True)

# Pasamos a formato ancho: un productor por fila, un tramo por columna
df_tramos_wide = df_tramos.reset_index().pivot(
    index="index0", columns="index1", values="p"
)
df_tramos_wide.columns.name = None

# Producción mínima efectiva
Pmin_df = P3.get_parameter("Pmin").get_values().to_pandas()
Pmin_series = Pmin_df.iloc[:, 0]
y_series = df_y["y.val"]
df_tramos_wide["Pmin"] = Pmin_series * y_series

# Producción total
df_tramos_wide["Total"] = (
    df_tramos_wide[["T1", "T2", "T3"]].sum(axis=1)
    + df_tramos_wide["Pmin"]
)

print(df_tramos_wide)
print(df_tramos_wide["Total"].sum())
valor_objetivo = P3.get_objective("coste_total").value()
print(f"\nValor objetivo (coste total): {valor_objetivo:.3f}")


     y.val
G1  1.0000
G2  1.0000
G3  0.5375
G4  0.0000
G5  0.0000
           T1      T2   T3    Pmin  Total
index0                                   
G1      20.00  20.000  0.0  10.000   50.0
G2       0.00   0.000  0.0   5.000    5.0
G3      33.75  16.875  0.0   5.375   56.0
G4       0.00   0.000  0.0   0.000    0.0
G5      90.00   0.000  0.0   0.000   90.0
201.0

Valor objetivo (coste total): 3162.000


este es P4 aunq se llame P3

In [ ]:
# Integración en Google Colab
from amplpy import AMPL, ampl_notebook

P3 = ampl_notebook(
    modules=["highs", "cplex"],  # Solvers que queremos instalar
    license_uuid="062f1a26-719d-4062-9a14-f91b8a2a0c4c",
)

Licensed to Bundle #7272.7817 expiring 20260228: 5003302 - Optimization; 5003408 - Operations Research, Prof. Juan Miguel Morales Gonz?lez, University of Malaga.


In [ ]:
P3.eval(r"""
reset;
set Prod;
set Tram;

param D >= 0;

param Pmin {Prod} >= 0;         # Producción mínima (zona muerta/prohibida)
param c    {Prod} >= 0;         # Coste fijo al producir

param lon   {Prod, Tram} >= 0;  # Longitud de cada tramo (variable)
param slope {Prod, Tram} >= 0;  # Pendiente de cada tramo (coste marginal)

var y {i in Prod} >=0,<=1;       # 1 si el productor i produce
var p {i in Prod, j in Tram} >= 0;  # Producción variable por tramo

minimize coste_total:
    sum{i in Prod} c[i]*y[i] + sum{i in Prod, j in Tram} slope[i,j]*p[i,j];

s.t. balance:
    sum{i in Prod} Pmin[i]*y[i] + sum{i in Prod, j in Tram} p[i,j] = D;
s.t. limite_tramo {i in Prod, j in Tram}:
    p[i,j] <= lon[i,j] * y[i];
bounding1: y["G2"]>=1;
bounding2: y["G5"]<=0;

""")

In [ ]:
# Cargar datos desde la función
demanda, tramos, productores, Pmin, c, lon_df, slope_df = preparar_datos_mercado_ext()

# Conjuntos
P3.set["Prod"] = productores
P3.set["Tram"] = tramos

# Parámetros
P3.get_parameter("D").set(demanda)
P3.get_parameter("Pmin").set_values(Pmin)
P3.get_parameter("c").set_values(c)

P3.get_parameter("lon").set_values(lon_df)
P3.get_parameter("slope").set_values(slope_df)



In [ ]:
P3.solve(solver="cplex") # Resolvemos con el solver "cplex"
assert P3.solve_result == "solved"  # Comprobamos que el problema se ha resuelto correctamente

CPLEX 22.1.2: CPLEX 22.1.2: optimal solution; objective 3162
6 simplex iterations


In [ ]:
import pandas as pd

y = P3.get_variable("y")
df_y = y.get_values().to_pandas()
print(df_y)

p = P1.get_variable("p")
df_p = p.get_values().to_pandas()



# Producción por tramos (tabla tal cual)
df_tramos = df_p.copy()
df_tramos.rename(columns={"p.val": "p"}, inplace=True)

# Pasamos a formato ancho: un productor por fila, un tramo por columna
df_tramos_wide = df_tramos.reset_index().pivot(
    index="index0", columns="index1", values="p"
)
df_tramos_wide.columns.name = None

# Producción mínima efectiva
Pmin_df = P3.get_parameter("Pmin").get_values().to_pandas()
Pmin_series = Pmin_df.iloc[:, 0]
y_series = df_y["y.val"]
df_tramos_wide["Pmin"] = Pmin_series * y_series

# Producción total
df_tramos_wide["Total"] = (
    df_tramos_wide[["T1", "T2", "T3"]].sum(axis=1)
    + df_tramos_wide["Pmin"]
)

print(df_tramos_wide)
print(df_tramos_wide["Total"].sum())
valor_objetivo = P3.get_objective("coste_total").value()
print(f"\nValor objetivo (coste total): {valor_objetivo:.3f}")


     y.val
G1  1.0000
G2  1.0000
G3  0.5375
G4  0.0000
G5  0.0000
           T1      T2   T3    Pmin  Total
index0                                   
G1      20.00  20.000  0.0  10.000   50.0
G2       0.00   0.000  0.0   5.000    5.0
G3      33.75  16.875  0.0   5.375   56.0
G4       0.00   0.000  0.0   0.000    0.0
G5      90.00   0.000  0.0   0.000   90.0
201.0

Valor objetivo (coste total): 3162.000


In [159]:
# Integración en Google Colab
from amplpy import AMPL, ampl_notebook

P6 = ampl_notebook(
    modules=["highs", "cplex"],  # Solvers que queremos instalar
    license_uuid="062f1a26-719d-4062-9a14-f91b8a2a0c4c",
)

Licensed to Bundle #7272.7817 expiring 20260228: 5003302 - Optimization; 5003408 - Operations Research, Prof. Juan Miguel Morales Gonz?lez, University of Malaga.


In [160]:
P6.eval(r"""
reset;
set Prod;
set Tram;

param D >= 0;

param Pmin {Prod} >= 0;         # Producción mínima (zona muerta/prohibida)
param c    {Prod} >= 0;         # Coste fijo al producir

param lon   {Prod, Tram} >= 0;  # Longitud de cada tramo (variable)
param slope {Prod, Tram} >= 0;  # Pendiente de cada tramo (coste marginal)

var y {i in Prod} >=0,<=1;       # 1 si el productor i produce
var p {i in Prod, j in Tram} >= 0;  # Producción variable por tramo

minimize coste_total:
    sum{i in Prod} c[i]*y[i] + sum{i in Prod, j in Tram} slope[i,j]*p[i,j];

s.t. balance:
    sum{i in Prod} Pmin[i]*y[i] + sum{i in Prod, j in Tram} p[i,j] = D;
s.t. limite_tramo {i in Prod, j in Tram}:
    p[i,j] <= lon[i,j] * y[i];
bounding1: y["G2"]<=0;
bounding2: y["G3"]>=1;

""")

In [161]:
# Cargar datos desde la función
demanda, tramos, productores, Pmin, c, lon_df, slope_df = preparar_datos_mercado_ext()

# Conjuntos
P6.set["Prod"] = productores
P6.set["Tram"] = tramos

# Parámetros
P6.get_parameter("D").set(demanda)
P6.get_parameter("Pmin").set_values(Pmin)
P6.get_parameter("c").set_values(c)

P6.get_parameter("lon").set_values(lon_df)
P6.get_parameter("slope").set_values(slope_df)



In [162]:
P6.solve(solver="cplex") # Resolvemos con el solver "cplex"
assert P6.solve_result == "solved"  # Comprobamos que el problema se ha resuelto correctamente

CPLEX 22.1.2: CPLEX 22.1.2: optimal solution; objective 3176
5 simplex iterations


In [163]:
import pandas as pd

y = P6.get_variable("y")
df_y = y.get_values().to_pandas()
print(df_y)

p = P6.get_variable("p")
df_p = p.get_values().to_pandas()



# Producción por tramos (tabla tal cual)
df_tramos = df_p.copy()
df_tramos.rename(columns={"p.val": "p"}, inplace=True)

# Pasamos a formato ancho: un productor por fila, un tramo por columna
df_tramos_wide = df_tramos.reset_index().pivot(
    index="index0", columns="index1", values="p"
)
df_tramos_wide.columns.name = None

# Producción mínima efectiva
Pmin_df = P6.get_parameter("Pmin").get_values().to_pandas()
Pmin_series = Pmin_df.iloc[:, 0]
y_series = df_y["y.val"]
df_tramos_wide["Pmin"] = Pmin_series * y_series

# Producción total
df_tramos_wide["Total"] = (
    df_tramos_wide[["T1", "T2", "T3"]].sum(axis=1)
    + df_tramos_wide["Pmin"]
)

print(df_tramos_wide)
print(df_tramos_wide["Total"].sum())
valor_objetivo = P6.get_objective("coste_total").value()
print(f"\nValor objetivo (coste total): {valor_objetivo:.3f}")


    y.val
G1      1
G2      0
G3      1
G4      0
G5      0
           T1      T2   T3  Pmin   Total
index0                                  
G1      20.00  20.000  0.0    10  50.000
G2       0.00   0.000  0.0     0   0.000
G3      33.75  16.875  0.0    10  60.625
G4       0.00   0.000  0.0     0   0.000
G5      90.00   0.000  0.0     0  90.000
200.625

Valor objetivo (coste total): 3176.000


In [154]:
# Integración en Google Colab
from amplpy import AMPL, ampl_notebook

P5 = ampl_notebook(
    modules=["highs", "cplex"],  # Solvers que queremos instalar
    license_uuid="062f1a26-719d-4062-9a14-f91b8a2a0c4c",
)

Licensed to Bundle #7272.7817 expiring 20260228: 5003302 - Optimization; 5003408 - Operations Research, Prof. Juan Miguel Morales Gonz?lez, University of Malaga.


In [155]:
P5.eval(r"""
reset;
set Prod;
set Tram;

param D >= 0;

param Pmin {Prod} >= 0;         # Producción mínima (zona muerta/prohibida)
param c    {Prod} >= 0;         # Coste fijo al producir

param lon   {Prod, Tram} >= 0;  # Longitud de cada tramo (variable)
param slope {Prod, Tram} >= 0;  # Pendiente de cada tramo (coste marginal)

var y {i in Prod} >=0,<=1;       # 1 si el productor i produce
var p {i in Prod, j in Tram} >= 0;  # Producción variable por tramo

minimize coste_total:
    sum{i in Prod} c[i]*y[i] + sum{i in Prod, j in Tram} slope[i,j]*p[i,j];

s.t. balance:
    sum{i in Prod} Pmin[i]*y[i] + sum{i in Prod, j in Tram} p[i,j] = D;
s.t. limite_tramo {i in Prod, j in Tram}:
    p[i,j] <= lon[i,j] * y[i];
bounding1: y["G2"]<=0;
bounding2: y["G3"]<=0;

""")

In [156]:
# Cargar datos desde la función
demanda, tramos, productores, Pmin, c, lon_df, slope_df = preparar_datos_mercado_ext()

# Conjuntos
P5.set["Prod"] = productores
P4.set["Tram"] = tramos

# Parámetros
P5.get_parameter("D").set(demanda)
P5.get_parameter("Pmin").set_values(Pmin)
P5.get_parameter("c").set_values(c)

P5.get_parameter("lon").set_values(lon_df)
P5.get_parameter("slope").set_values(slope_df)



In [157]:
P5.solve(solver="cplex") # Resolvemos con el solver "cplex"
assert P5.solve_result == "solved"  # Comprobamos que el problema se ha resuelto correctamente

CPLEX 22.1.2: CPLEX 22.1.2: optimal solution; objective 3149.214286
6 simplex iterations


In [158]:
import pandas as pd

y = P5.get_variable("y")
df_y = y.get_values().to_pandas()
print(df_y)

p = P5.get_variable("p")
df_p = p.get_values().to_pandas()



# Producción por tramos (tabla tal cual)
df_tramos = df_p.copy()
df_tramos.rename(columns={"p.val": "p"}, inplace=True)

# Pasamos a formato ancho: un productor por fila, un tramo por columna
df_tramos_wide = df_tramos.reset_index().pivot(
    index="index0", columns="index1", values="p"
)
df_tramos_wide.columns.name = None

# Producción mínima efectiva
Pmin_df = P5.get_parameter("Pmin").get_values().to_pandas()
Pmin_series = Pmin_df.iloc[:, 0]
y_series = df_y["y.val"]
df_tramos_wide["Pmin"] = Pmin_series * y_series

# Producción total
df_tramos_wide["Total"] = (
    df_tramos_wide[["T1", "T2", "T3"]].sum(axis=1)
    + df_tramos_wide["Pmin"]
)

print(df_tramos_wide)
print(df_tramos_wide["Total"].sum())
valor_objetivo = P5.get_objective("coste_total").value()
print(f"\nValor objetivo (coste total): {valor_objetivo:.3f}")


       y.val
G1  1.000000
G2  0.000000
G3  0.000000
G4  0.385714
G5  1.000000
           T1      T2   T3       Pmin      Total
index0                                          
G1      20.00  20.000  0.0  10.000000  50.000000
G2       0.00   0.000  0.0   0.000000   0.000000
G3      33.75  16.875  0.0   0.000000  50.625000
G4       0.00   0.000  0.0   1.928571   1.928571
G5      90.00   0.000  0.0   7.000000  97.000000
199.55357142857144

Valor objetivo (coste total): 3149.214
